In [71]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure Display Options
pd.set_option('display.max_colwidth', None)

# Configure Plotting Style
sns.set_theme(style="whitegrid")
%matplotlib inline

# Camera Dataset Analysis

In [105]:
# Define Path to data directory and camera data
data_dir = os.path.join("..", "Data")
json_path = os.path.join(data_dir, "camera_data.json")

# Print the absolute path to verify
print(f"Reading data from: {os.path.abspath(data_dir)}\n")

# Load the data
with open(json_path, 'r') as f:
    camera_data = json.load(f)

# Verify the outer container is a list
print(f"Type of camera_data: {type(camera_data)}") 

# Verify the items inside are dictionaries
first_item = camera_data[0]
print(f"Type of first item:  {type(first_item)}")

# View the keys of that ditionary
print(f"Keys available:      {first_item.keys()}")

Reading data from: c:\Users\ThinkPad\Desktop\Repositories\BDCM\liesel_project\PackageComparison\Data

Type of camera_data: <class 'list'>
Type of first item:  <class 'dict'>
Keys available:      dict_keys(['y', 'X'])


In [118]:
# Select the first respondent to analyze the structure
# (Assuming all respondents have the same design structure)
respondent = camera_data[0]
y_sample = respondent['y']
X_sample = respondent['X']

# 1. Total Number of Decision Making Units (N)
n_units = len(camera_data)

# 2. Number of Decisions per Unit (T)
n_decisions_per_unit = len(y_sample)

# 3. Number of Alternatives per Decision (J)
# The design matrix X has (Decisions * Alternatives) rows.
# So: 80 rows / 16 decisions = 5 alternatives.
n_alternatives = int(len(X_sample) / len(y_sample))

# 4. Total Number of Decisions (N * T)
total_decisions = sum(len(resp['y']) for resp in camera_data)

# --- Print the Summary ---
print(f"Number of Alternatives per Decision (J):     {n_alternatives}")
print(f"Total number of Decision Making Units (N):   {n_units}")
print(f"Number of Decisions per Unit (T):            {n_decisions_per_unit}")
print(f"Total Number of Decisions (N*T):             {total_decisions}")

Number of Alternatives per Decision (J):     5
Total number of Decision Making Units (N):   332
Number of Decisions per Unit (T):            16
Total Number of Decisions (N*T):             5312


First, we isolate a single person (respondent 1) to take a look at the dataset

In [108]:
# Select Respondent 1 (Index 0)
respondent = camera_data[0]

#Extract their components
y = respondent['y']           # The choices they made
X = np.array(respondent['X']) # The design matrix

print(f"Number of Decisions (y):    {len(y)}")
print(f"Shape of Design Matrix (X): {X.shape}")

print(f"First 5 choices (y): {y[:5]}")
print(f"First 5 rows of X:\n{X[:25]}")

Number of Decisions (y):    16
Shape of Design Matrix (X): (80, 10)
First 5 choices (y): [1, 2, 2, 4, 2]
First 5 rows of X:
[[0.   0.   1.   0.   0.   1.   0.   1.   0.   0.79]
 [1.   0.   0.   0.   1.   1.   0.   1.   1.   2.29]
 [0.   0.   0.   1.   0.   0.   0.   0.   1.   1.29]
 [0.   1.   0.   0.   0.   1.   0.   1.   0.   2.79]
 [0.   0.   0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   1.   1.   0.   0.   1.   0.   2.79]
 [1.   0.   0.   0.   1.   0.   1.   0.   1.   0.79]
 [0.   1.   0.   0.   1.   0.   1.   0.   1.   1.79]
 [0.   0.   1.   0.   1.   0.   1.   1.   1.   1.29]
 [0.   0.   0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   1.   0.   0.   0.   0.   0.   1.   2.29]
 [1.   0.   0.   0.   1.   1.   0.   1.   1.   0.79]
 [0.   1.   0.   0.   1.   0.   0.   1.   1.   1.79]
 [0.   1.   0.   0.   1.   1.   1.   0.   1.   1.29]
 [0.   0.   0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   1.   0.   1.   1.   0.   1.   1.   1.79]
 [0.   1.   0.   0.   0.   0

# Understanding the `bayesm` Camera Dataset Structure

The `camera` dataset is originally stored as a hierarchical list (R), was exported as a Json and is therefore loaded as a list of dictionaries (Python). To analyze it correctly, we must understand how the Choice Vector ($y$) interacts with the Design Matrix ($X$), as well as specific traits in the survey design.

### Key Concepts

1.  **The Design Matrix ($X$):** Contains 80 rows for each respondent.
    * There are **16 Choice Tasks**.
    * Each task has **5 Alternatives**.
    * $16 \times 5 = 80$ rows total.
    * **CRITICAL (Randomization):** The order of brands within each choice task is **randomized (shuffled)**. You cannot assume Row 1 is always Canon. You must check the dummy coded brand feature columns (Indices 0-3) to identify the brand in a specific row.

2.  **The Choice Vector ($y$):** Contains 16 integers (e.g., `[1, 2, ...]`).
    * These numbers represent the **row index (position)** of the chosen alternative in the specific block of 5 options.
    * **$y=1$:** The user chose the **1st option** shown in the list (Top Row).
    * **$y=2$:** The user chose the **2nd option** shown in the list (Second Row).
    * ...
    * **$y=5$:** The user chose the **5th option**. Note that while the 5th option is typically the "None" option, the "None" option (all zeros) can technically appear in other rows if a brand is missing.

3.  **Survey Design Peculiarities (Important Findings):**
    * **Partial Profile (Missing Brands):** This is *not* a standard Balanced Design. In some tasks, specific brands are **intentionally missing** (unavailable). In these cases, you may see multiple rows of zeros, or simply a brand dummy column that never equals `1` in that task block.
    * **Intra-Brand Competition (Duplicate Brands):** Some choice tasks present the **same brand multiple times** (e.g., Row 1 is Canon, and Row 3 is *also* Canon). This structure allows the model to measure trade-offs *within* a brand (e.g., a cheap Canon vs. an expensive Canon) rather than just *between* brands.

### How to Decode a Choice
To determine precisely what a user bought, you must follow this 2-step lookup:

1.  **Read $y$:** See which row position was clicked (e.g., `y=1`).
2.  **Read $X$:** Go to that specific row in the $X$ matrix and inspect the brand columns:
    * **Brand Check:** Check which column (0=Canon, 1=Sony, 2=Nikon, 3=Panasonic) is `1`.
    * **None Check:** If **all** brand columns are `0`, the user selected the "None" / "No Purchase" option.

**Example:**
If `y=1` and the first row of $X$ has `Nikon=1`, then the user chose **Nikon**, even though the index is 1.

In [96]:
# SETUP & DATA EXTRACTION
X = np.array(respondent['X'])
y = respondent['y']

# Define feature names and expected brand set
features = ["canon", "sony", "nikon", "panasonic", "pixels", "zoom", "video", "swivel", "wifi", "price"]
brands_expected = {"Canon", "Sony", "Nikon", "Panasonic", "None"}


#  HELPER FUNCTIONS
def get_brand_label(row_data):
    """
    Identifies the brand from a single row of the design matrix (dummy columns).
    """
    if row_data[0] == 1: return "Canon"
    if row_data[1] == 1: return "Sony"
    if row_data[2] == 1: return "Nikon"
    if row_data[3] == 1: return "Panasonic"
    
    # Check if all brand columns are 0 (strictly)
    if np.all(row_data[:4] == 0):
        return "None"
    
    return "Unknown"

def analyze_availability(X_matrix):
    """
    Iterates through all 16 tasks to identify missing brands (Partial Profile).
    """
    report_rows = []
    
    for task_idx in range(16):
        # Slice the 5 rows for this task
        start_row = task_idx * 5
        end_row = start_row + 5
        task_block = X_matrix[start_row:end_row]
        
        # Identify content of each row in this block
        present_in_task = set()
        row_descriptions = []
        
        for row in task_block:
            label = get_brand_label(row)
            present_in_task.add(label)
            row_descriptions.append(label)

        # Check for Missing Brands
        missing_items = brands_expected - present_in_task
        is_complete = len(missing_items) == 0
        
        report_rows.append({
            "Task_ID": task_idx + 1,
            "Is_Complete": is_complete,
            "Missing_Brands": list(missing_items) if missing_items else "Full Set",
            "Row_Structure": row_descriptions
        })

    return pd.DataFrame(report_rows)

def display_task_details(X_matrix, y_vector, task_idx, title_suffix=""):
    """
    Visualizes a specific choice task and the user's decision.
    """
    start_row = task_idx * 5
    end_row = start_row + 5
    task_block = X_matrix[start_row:end_row]
    
    # Create DataFrame for display
    df_task = pd.DataFrame(task_block, columns=features)
    
    # Generate dynamic row labels
    row_labels = [f"Option {i+1}: {get_brand_label(row)}" for i, row in enumerate(task_block)]
    df_task.index = row_labels
    
    print(f"\n--- {title_suffix} (Choice Task {task_idx + 1}) ---")
    display(df_task)
    
    # Show User Choice
    choice_idx = y_vector[task_idx] 
    chosen_label = df_task.index[choice_idx - 1]
    print(f"User Choice Index: {choice_idx}")
    print(f"User Selected: {chosen_label}")


# EXECUTION & ANALYSIS

# Print Raw Choice Vector
print(f"Number of Decisions:    {len(y)}")
print(f"Respondent Choices:     {y}")

# Run Availability Analysis
print("\n=== AVAILABILITY ANALYSIS (Partial Profile Check) ===")
df_availability = analyze_availability(X)
incomplete_tasks = df_availability[df_availability['Is_Complete'] == False]

if not incomplete_tasks.empty:
    print(f"Found {len(incomplete_tasks)} tasks with missing options:")
    display(incomplete_tasks[['Task_ID', 'Missing_Brands', 'Row_Structure']])
else:
    print("All tasks were complete (Full Profile).")

# Visualize Specific Tasks (Task 1 & Task 2)
print("\n=== INDIVIDUAL TASK INSPECTION ===")

# Task 1
display_task_details(X, y, task_idx=0, title_suffix="Available Options")

# Task 2
display_task_details(X, y, task_idx=1, title_suffix="Available Options")

Number of Decisions:    16
Respondent Choices:     [1, 2, 2, 4, 2, 2, 1, 1, 1, 2, 3, 2, 1, 2, 3, 1]

=== AVAILABILITY ANALYSIS (Partial Profile Check) ===
Found 5 tasks with missing options:


,Task_ID,Missing_Brands,Row_Structure
2,3,[Panasonic],"[Nikon, Canon, Sony, Sony, None]"
3,4,[Canon],"[Nikon, Sony, Panasonic, Panasonic, None]"
4,5,[Nikon],"[Canon, Canon, Sony, Panasonic, None]"
7,8,[Canon],"[Panasonic, Sony, Nikon, Nikon, None]"
10,11,[Sony],"[Nikon, Canon, Panasonic, Panasonic, None]"



=== INDIVIDUAL TASK INSPECTION ===

--- Available Options (Choice Task 1) ---


,canon,sony,nikon,panasonic,pixels,zoom,video,swivel,wifi,price
Option 1: Nikon,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.79
Option 2: Canon,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,2.29
Option 3: Panasonic,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.29
Option 4: Sony,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,2.79
Option 5: None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00


User Choice Index: 1
User Selected: Option 1: Nikon

--- Available Options (Choice Task 2) ---


,canon,sony,nikon,panasonic,pixels,zoom,video,swivel,wifi,price
Option 1: Panasonic,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,2.79
Option 2: Canon,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.79
Option 3: Sony,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.79
Option 4: Nikon,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,1.29
Option 5: None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00


User Choice Index: 2
User Selected: Option 2: Canon


In [130]:
# 1. Define the desired order
order_list = ["Canon", "Sony", "Nikon", "Panasonic", "None"]

# Initialize a counter for brands
brand_counts = {brand: 0 for brand in brands_expected}

# Iterate through every respondent in the dataset
for respondent in camera_data:
    X_resp = np.array(respondent['X'])
    y_resp = respondent['y']
    
    # Iterate through each of the 16 tasks per respondent
    for task_idx, choice_idx in enumerate(y_resp):
        # Calculate the specific row index in X
        chosen_row_idx = (task_idx * 5) + (choice_idx - 1)
        chosen_row_data = X_resp[chosen_row_idx]
        
        # Identify the brand and update the counter
        brand_name = get_brand_label(chosen_row_data)
        brand_counts[brand_name] += 1

# 2. Convert to DataFrame and REINDEX immediately to set the order
df_shares = pd.DataFrame.from_dict(brand_counts, orient='index', columns=['Total_Choices'])
df_shares.index.name = 'Brand'
df_shares = df_shares.reindex(order_list)

# 3. Calculate Relative Share (%)
# We keep this as a float for a moment to ensure precision
df_shares['Share_Percentage'] = (df_shares['Total_Choices'] / total_decisions) * 100

# 4. Final Formatting: 2 decimal places and the % symbol
df_shares['Share_Percentage'] = df_shares['Share_Percentage'].map("{:.1f}%".format)

print("=== BRAND CHOICE SUMMARY ===")
display(df_shares)

=== BRAND CHOICE SUMMARY ===


,Total_Choices,Share_Percentage
Brand,,
Canon,1108,20.9%
Sony,975,18.4%
Nikon,1058,19.9%
Panasonic,828,15.6%
None,1343,25.3%


##  Insight: Brand Share vs. Position Share

When comparing our Python brand-mapping results with the standard [bayesm Overview Vignette](https://cran.r-project.org/web/packages/bayesm/vignettes/bayesm_Overview_Vignette.html#4_Examples) documentation and R output, we find a critical distinction in how choices are interpreted.

### Position-Based Analysis (R)
The following R code calculates the frequency of the **chosen row index (Position)** across all respondents:

```r
N <- length(camera)
dat <- matrix(NA, N*16, 2)
for (i in 1:length(camera)) {
  Ni <- length(camera[[i]]$y)
  dat[((i-1)*Ni+1):(i*Ni),1] <- i
  dat[((i-1)*Ni+1):(i*Ni),2] <- camera[[i]]$y
}
round(prop.table(table(dat[,2])), 3)

## 
##     1     2     3     4     5 
## 0.207 0.176 0.195 0.169 0.253

```

### Brand-Based Analysis (Python)
The Python logic maps the choice index ($y$) back to the design matrix ($X$) to identify the **actual brand**.
* **What it measures:** The true market share of Canon, Sony, Nikon, and Panasonic.
* **Finding:** Canon has a market share of **20.86%**.
* **Use Case:** Marketing and competitive analysis.

### Why they don't match exactly
Because the brand order is **randomized** across the 16 tasks:
* Canon is not always in Position 1.
* In some tasks, Canon might be in Position 3, or even missing entirely (Partial Profile).
* The only consistent metric is the **"None" option (Position 5)**, which yielded nearly identical results in both environments (~25.3%).

**Conclusion:** The Python brand-mapping logic should be the correct approach for estimating brand preferences, as it accounts for the experiment design.
